# Dataset Variable Summary

This notebook reads the parsed summary file and presents a human-readable overview for each dataset:
- rows and columns
- missingness
- dominant data type
- numeric ranges (where available)
- example values

In [ ]:
import json
from pathlib import Path

summary_candidates = [
    Path("dataset_summary.json"),
    Path("dataset_summary.sample200k.json"),
]

summary_path = next((p for p in summary_candidates if p.exists()), None)
if summary_path is None:
    raise FileNotFoundError(
        "No summary JSON found. Expected dataset_summary.json or dataset_summary.sample200k.json"
    )

summaries = json.loads(summary_path.read_text(encoding="utf-8"))
print(f"Using summary file: {summary_path}\n")

for ds in summaries:
    print("=" * 80)
    print(f"Dataset: {ds['file']}")
    print(f"Rows: {ds['rows']:,}")
    print(f"Columns: {ds['columns']}")

    variables = ds.get("variables", [])

    # Top missing columns
    top_missing = sorted(
        variables,
        key=lambda v: v.get("missing_ratio", 0),
        reverse=True,
    )[:5]

    print("\nTop columns by missing ratio:")
    for v in top_missing:
        print(
            f"- {v['column']}: {v.get('missing_ratio', 0):.2%} missing "
            f"({v.get('null_count', 0):,}/{ds['rows']:,})"
        )

    print("\nPer-column summary:")
    for v in variables:
        rng = ""
        if v.get("numeric_min") is not None and v.get("numeric_max") is not None:
            rng = f", range=[{v['numeric_min']}, {v['numeric_max']}]"

        examples = ", ".join(v.get("examples", [])[:3])
        if not examples:
            examples = "(no examples)"

        print(
            f"- {v['column']}: type={v.get('dominant_type')}, "
            f"missing={v.get('missing_ratio', 0):.2%}{rng}, "
            f"examples: {examples}"
        )

    print()

: 

In [2]:
import json
from pathlib import Path


def r_type(dominant_type: str) -> str:
    mapping = {
        "int": "int",
        "float": "num",
        "bool": "logi",
        "date": "chr",
        "time": "chr",
        "text": "chr",
        None: "chr",
    }
    return mapping.get(dominant_type, "chr")


def fmt_examples(vals):
    if not vals:
        return ""
    shown = vals[:3]
    return " c(" + ", ".join(f'\"{x}\"' for x in shown) + ")"


summary_candidates = [
    Path("dataset_summary.json"),
    Path("dataset_summary.sample200k.json"),
]
summary_path = next((p for p in summary_candidates if p.exists()), None)
if summary_path is None:
    raise FileNotFoundError("No summary JSON found.")

summaries = json.loads(summary_path.read_text(encoding="utf-8"))

print(f"List of {len(summaries)}")
for ds in summaries:
    file_name = ds["file"]
    rows = ds["rows"]
    cols = ds["columns"]
    vars_ = ds.get("variables", [])

    print(f"$ {file_name}: 'data.frame':\t{rows:,} obs. of  {cols} variables:")

    # Show all columns in R-like str() style
    for v in vars_:
        col = v.get("column", "unknown")
        r_t = r_type(v.get("dominant_type"))
        missing = v.get("missing_ratio", 0)
        na_note = f", NA={missing:.2%}" if missing else ""

        if v.get("numeric_min") is not None and v.get("numeric_max") is not None:
            rng = f" [{v['numeric_min']}, {v['numeric_max']}]"
        else:
            rng = ""

        ex = fmt_examples(v.get("examples", []))
        print(f" ..$ {col}: {r_t}{rng}{na_note}{ex}")

    print()

List of 3
$ games.csv.gz: 'data.frame':	200,001 obs. of  23 variables:
 ..$ tournamentid: chr c("7iv6zDwO", "33LR4dZi", "yLl4X7yk")
 ..$ game_id: chr c("7iv6zDwO-0", "7iv6zDwO-1", "7iv6zDwO-2")
 ..$ event: chr c("≤1700 SuperBlitz Arena", "Hourly SuperBlitz Arena", "Hourly Blitz Arena")
 ..$ site: chr c("https://lichess.org/Bqy0EAxX", "https://lichess.org/yvp5kfrU", "https://lichess.org/nMEP2h1W")
 ..$ date: chr c("2026.04.14", "2026.04.13", "2026.04.12")
 ..$ round: chr c("-")
 ..$ white: chr c("pedrovalentinc", "meakash", "BlackRockVanguard")
 ..$ black: chr c("Abu-suhan", "bong_jovi", "SdoTdoU")
 ..$ result: chr c("0-1", "1-0", "1/2-1/2")
 ..$ utcdate: chr c("2026.04.14", "2026.04.13", "2026.04.12")
 ..$ utctime: chr c("11:54:29", "11:53:56", "11:53:23")
 ..$ whiteelo: int [408.0, 3034.0] c("1330", "1578", "1618")
 ..$ blackelo: int [400.0, 3033.0] c("1657", "1471", "1636")
 ..$ whiteratingdiff: int [-160.0, 519.0], NA=0.01% c("-2", "+8", "-5")
 ..$ blackratingdiff: int [-182.0, 368.

In [3]:
# R str()-style summary for moves only

def r_type(dominant_type: str) -> str:
    mapping = {
        "int": "int",
        "float": "num",
        "bool": "logi",
        "date": "chr",
        "time": "chr",
        "text": "chr",
        None: "chr",
    }
    return mapping.get(dominant_type, "chr")


def fmt_examples(vals):
    if not vals:
        return ""
    shown = vals[:3]
    return " c(" + ", ".join(f'\"{x}\"' for x in shown) + ")"


moves_ds = next((d for d in summaries if d.get("file") == "moves.csv.gz"), None)
if moves_ds is None:
    print("moves.csv.gz was not found in the loaded summary.")
else:
    file_name = moves_ds["file"]
    rows = moves_ds["rows"]
    cols = moves_ds["columns"]
    vars_ = moves_ds.get("variables", [])

    print("List of 1")
    print(f"$ {file_name}: 'data.frame':\t{rows:,} obs. of  {cols} variables:")

    for v in vars_:
        col = v.get("column", "unknown")
        r_t = r_type(v.get("dominant_type"))
        missing = v.get("missing_ratio", 0)
        na_note = f", NA={missing:.2%}" if missing else ""

        if v.get("numeric_min") is not None and v.get("numeric_max") is not None:
            rng = f" [{v['numeric_min']}, {v['numeric_max']}]"
        else:
            rng = ""

        ex = fmt_examples(v.get("examples", []))
        print(f" ..$ {col}: {r_t}{rng}{na_note}{ex}")

List of 1
$ moves.csv.gz: 'data.frame':	200,001 obs. of  5 variables:
 ..$ game_id: chr c("7iv6zDwO-0", "7iv6zDwO-1", "7iv6zDwO-2")
 ..$ move_no: int [1.0, 86.0] c("1", "2", "3")
 ..$ move: chr c("e4", "c6", "d4")
 ..$ clock: chr c("0:03:00", "0:02:57", "0:02:56")
 ..$ color: chr c("white", "black")


In [4]:
# R str()-style summary for tournaments only

def r_type(dominant_type: str) -> str:
    mapping = {
        "int": "int",
        "float": "num",
        "bool": "logi",
        "date": "chr",
        "time": "chr",
        "text": "chr",
        None: "chr",
    }
    return mapping.get(dominant_type, "chr")


def fmt_examples(vals):
    if not vals:
        return ""
    shown = vals[:3]
    return " c(" + ", ".join(f'\"{x}\"' for x in shown) + ")"


tournaments_ds = next((d for d in summaries if d.get("file") == "tournaments.csv.gz"), None)
if tournaments_ds is None:
    print("tournaments.csv.gz was not found in the loaded summary.")
else:
    file_name = tournaments_ds["file"]
    rows = tournaments_ds["rows"]
    cols = tournaments_ds["columns"]
    vars_ = tournaments_ds.get("variables", [])

    print("List of 1")
    print(f"$ {file_name}: 'data.frame':\t{rows:,} obs. of  {cols} variables:")

    for v in vars_:
        col = v.get("column", "unknown")
        r_t = r_type(v.get("dominant_type"))
        missing = v.get("missing_ratio", 0)
        na_note = f", NA={missing:.2%}" if missing else ""

        if v.get("numeric_min") is not None and v.get("numeric_max") is not None:
            rng = f" [{v['numeric_min']}, {v['numeric_max']}]"
        else:
            rng = ""

        ex = fmt_examples(v.get("examples", []))
        print(f" ..$ {col}: {r_t}{rng}{na_note}{ex}")

List of 1
$ tournaments.csv.gz: 'data.frame':	53,471 obs. of  32 variables:
 ..$ id: chr c("7iv6zDwO", "yLl4X7yk", "33LR4dZi")
 ..$ createdBy: chr c("lichess", "cormacobear", "aaarmstark")
 ..$ system: chr c("arena")
 ..$ minutes: int [27.0, 360.0] c("57", "27", "60")
 ..$ rated: logi c("True", "False")
 ..$ fullName: chr c("≤1700 SuperBlitz Arena", "Hourly Blitz Arena", "Hourly SuperBlitz Arena")
 ..$ nbPlayers: int [351.0, 3458.0] c("379", "407", "815")
 ..$ startsAt: int [1694701898108.0, 1776164420000.0] c("1776164420000", "1776164415000", "1776164405000")
 ..$ finishesAt: int [1694703518108.0, 1776167840000.0] c("1776167840000", "1776167835000", "1776167825000")
 ..$ hasMaxRating: logi, NA=73.27% c("True")
 ..$ position: chr, NA=97.79% c("{'eco': 'C01', 'name': 'French Defense: Exchange Variation', 'fen': 'rnbqkbnr/ppp2ppp/4p3/3P4/3P4/8/PPP2PPP/RNBQKBNR b KQkq -', 'url': 'https://lichess.org/opening/French_Defense_Exchange_Variation'}", "{'eco': 'D06', 'name': "Queen's Gambit", 'f

In [6]:
# Parse games.csv.gz and count unique values in the `event` column
import csv
import gzip
from pathlib import Path

file_path = Path("games.csv.gz")
if not file_path.exists():
    raise FileNotFoundError("games.csv.gz not found in the current working directory.")

unique_events = set()

with gzip.open(file_path, mode="rt", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        event_value = (row.get("event") or "").strip()
        if event_value:
            unique_events.add(event_value)

print(f"Unique values in games.csv.gz -> event: {len(unique_events):,}")

Unique values in games.csv.gz -> event: 155


In [ ]:
from pathlib import Path
import csv
import gzip

base = Path.cwd()
candidates = [base / "moves.csv.gz", base / "moves.csv"]

moves_path = next((p for p in candidates if p.exists()), None)
if moves_path is None:
    raise FileNotFoundError("Could not find moves.csv.gz or moves.csv in the current directory")

if moves_path.suffix == ".gz":
    with gzip.open(moves_path, mode="rt", encoding="utf-8", newline="") as f:
        row_count = sum(1 for _ in csv.reader(f)) - 1  # minus header
else:
    with open(moves_path, mode="rt", encoding="utf-8", newline="") as f:
        row_count = sum(1 for _ in csv.reader(f)) - 1  # minus header

print(f"Rows in {moves_path.name}: {row_count:,}")

: 